In [1]:
import pandas as pd
import numpy as np
import os
import csv
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

mapping = {
"iso3c":"COUNTRY_CODE",
"country": "COUNTRY_LABEL",
"item_code":"COMMODITY_CODE",
"item_en":"COMMODITY_LABEL",
"year": "YEAR",
"trade":"TRADE_STATUS",
"bmkp_lcu_mt":"BP",
"forex_o":"ER_USED",
"r_pcp_mt":"RP",
"bpc_qt":"RP_POC_QADJ",
"bpc_ql":"RP_POC_QLTADJ",
"bpc_mrg_mt":"RP_POC_PM",
"bpc_trsp_mt":"RP_POC_TM",
"bpc_other_mt":"RP_POC_OC",
"r_pcp_mt":"POCPRICE",
"pcfg_mrg_mt":"POC_FARM_PM",
"pcfg_trsp_mt":"POC_FARM_TM",
"pcfg_other_mt":"POC_FARM_OC",
"pcfg_qt":"POC_FARM_QADJ",
"pcfg_ql":"POC_FARM_QLTADJ",
"fgp":"PROP",
"r_fgp_c":"REFP",
"bot":"PSCT",
"nrp_fg_o":'NPC_SOURCE'
}

base_dir = os.getcwd()

In [2]:
exch = pd.read_csv(os.path.join(base_dir, "Exchange_Rate_2024.csv"), nrows=24)
exch = exch.drop(['Series Name','Series Code', 'Country Name'],axis=1)
exch.rename(columns={'Country Code':'COUNTRY_CODE'}, inplace=True)
exch = pd.melt(exch,id_vars=['COUNTRY_CODE'],var_name='YEAR', value_name='EXCH')
exch['YEAR'] = exch['YEAR'].apply(lambda x: x.split(' ')[0]).astype(int)
exch = exch.sort_values(['COUNTRY_CODE','YEAR'])
exch.head()

,COUNTRY_CODE,YEAR,EXCH
17,ARM,2005,457.686941
41,ARM,2006,416.040370
65,ARM,2007,342.079116
89,ARM,2008,305.969400
113,ARM,2009,363.283286


In [3]:
# eth_exch = pd.read_excel(os.path.join(base_dir, "Ethiopia_Exchange_Rate.xlsx"), nrows=2)
# eth_exch = eth_exch.drop(['Country Name'],axis=1)
# eth_exch.rename(columns={'Country Code':'COUNTRY_CODE'}, inplace=True)
# eth_exch = pd.melt(eth_exch,id_vars=['COUNTRY_CODE'],var_name='YEAR', value_name='EXCH')
# eth_exch['YEAR'] = eth_exch['YEAR'].apply(lambda x: x.split(' ')[0]).astype(int)
# eth_exch.head()

In [4]:
# droping WDI data for Ethiopia. Using exchange rate from MAFAP database 
# exch = exch[exch['COUNTRY_CODE']!='ETH']
# exch = exch.append(eth_exch)
# exch.head()

In [5]:
data = pd.read_excel(os.path.join(base_dir,"PI_consortium_dataset_2025-02-26_upd.xlsx"))
data = data.merge(exch,how="left",right_on=["COUNTRY_CODE","YEAR"],left_on=["iso3c","year"])
data = data[data.item_en!='Fertilizer']
data.bpc_other_o = data.bpc_other_o.astype(float)
del data['YEAR']
del data['COUNTRY_CODE']
print(data.shape)
data = data[data.nrp_fg_o.notnull()]
print(data.shape)
data.head()

(1743, 75)
(1723, 75)


,country,year,trade,foodsec,item_en,bmkp_o,bmkp_a,forex_o,forex_a,bpc_ac_o,bpc_ac_a,bpc_trsp_o,bpc_trsp_a,bpc_mrg_o,bpc_mrg_a,bpc_prs_o,bpc_prs_a,bpc_hnd_o,bpc_hnd_a,bpc_taxs_o,bpc_taxs_a,bpc_other_o,bpc_other_a,pcp,pcfg_ac_o,pcfg_ac_a,pcfg_trsp_o,pcfg_trsp_a,pcfg_mrg_o,pcfg_mrg_a,pcfg_prs_o,pcfg_prs_a,pcfg_hnd_o,pcfg_hnd_a,pcfg_taxs_o,pcfg_taxs_a,pcfg_other_o,pcfg_other_a,fgp,bot,bpc_qt,bpc_ql,pcfg_qt,pcfg_ql,pcrt_qt,pcrt_ql,bmkp_lcu_o,bmkp_lcu_a,r_pcp_o,r_pcp_a,r_fgp_o,r_fgp_a,pg_pc_o,pg_pc_a,pg_fg_o,pg_fg_a,pg_rt_o,pg_rt_a,nrp_pc_o,nrp_pc_a,nrp_fg_o,nrp_fg_a,nra_o,nra_a,img,forexg,acg_pc,acg_fg,mdg,mdg_percent_o,mdg_percent_a,iso3c,item_fr,item_code,EXCH
0,Bangladesh,2014,m,y,Onions,302.762385,302.762385,77.641,77.641,2396.162321,1197.818032,45.484884,22.479314,2350.677437,1175.338719,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30500.0,5254.303864,4118.999589,757.042931,374.142001,1834.558210,1160.469652,NaN,NaN,1200.826718,1200.826718,365.469001,287.154215,1096.407003,1096.407003,28125.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,23506.77437,23506.77437,25902.93669,24704.59240,20648.63283,20585.59281,4597.063309,5795.407597,7476.367172,7539.407187,NaN,NaN,17.747267,23.458828,36.207565,36.624680,36.207565,36.624680,0.0,0.0,1198.344289,-1135.304274,63.040014,0.224142,0.306234,BGD,Oignon,402,77.641408
1,Bangladesh,2015,m,y,Onions,434.938053,434.938053,77.947,77.947,3438.607555,1719.023809,48.395916,23.917990,3390.211639,1695.105819,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38715.0,5590.579311,4721.163243,805.493679,398.087089,1951.969935,1573.287389,NaN,NaN,1277.679628,1277.679628,388.859017,305.532085,1166.577052,1166.577052,38130.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,33902.11639,33902.11639,37340.72394,35621.14020,31750.14463,30899.97696,1374.276055,3093.859802,6379.855366,7230.023044,NaN,NaN,3.680368,8.685460,20.093941,23.398149,20.093941,23.398149,0.0,0.0,1719.583746,-869.416068,850.167678,2.229656,2.751354,BGD,Oignon,402,77.946908
2,Bangladesh,2016,m,y,Onions,201.448763,201.448763,78.468,78.468,1631.989105,820.603937,51.260955,30.239861,1580.728150,790.364075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26925.0,5921.541606,4531.172242,853.178904,503.307282,2067.526555,1115.288700,NaN,NaN,1353.318262,1353.318262,411.879471,323.619584,1235.638413,1235.638413,27030.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,15807.28150,15807.28150,17439.27061,16627.88544,11517.72900,12096.71320,9485.729392,10297.114560,15512.271000,14933.286800,NaN,NaN,54.392925,61.926785,134.681686,123.449127,134.681686,123.449127,0.0,0.0,811.385168,-1390.369364,-578.984196,-2.142006,-4.786294,BGD,Oignon,402,78.468092
3,Bangladesh,2017,m,y,Onions,234.200000,234.200000,80.438,80.438,1937.907511,973.813890,54.049551,31.884910,1883.857960,941.928980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32440.0,6243.673469,4419.297159,899.591837,530.687199,2180.000000,817.589552,NaN,NaN,1426.938776,1426.938776,434.285714,341.224490,1302.857143,1302.857143,19815.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,18838.57960,18838.57960,20776.48711,19812.39349,14532.81364,15393.09633,11663.512890,12627.606510,5282.186359,4421.903669,NaN,NaN,56.138042,63.735897,36.346619,28.726538,36.346619,28.726538,0.0,0.0,964.093621,-1824.376311,-860.282690,-4.341573,-5.588757,BGD,Oignon,402,80.437542
4,Bangladesh,2018,m,y,Onions,251.100000,251.100000,83.466,83.466,2153.004875,1081.522084,57.173615,33.606454,2095.831260,1047.915630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36615.0,6604.557796,5254.275054,951.588245,559.340292,2306.004000,1446.409374,NaN,NaN,1509.415837,1509.415837,459.387429,360.947265,1378.162286,1378.162286,35055.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,20958.31260,20958.31260,23111.31747,22039.83468,16506.75968,16785.55963,13503.682530,14575.165320,18548.240320,18269.440370,NaN,NaN,58.428875,66.131012,112.367546,108.840225,112.367546,108.840225,0.0,0.0,1071.482791,-1350.282742,-278.799951,-0.795322,-1.660951,BGD,Oignon,402,83.466202


In [6]:
newdata = pd.read_excel(os.path.join(base_dir, "Data EECCA for AgIncentives 20221115.xlsx"), skiprows=[0])
newdata = newdata.merge(exch,how="left",right_on=["COUNTRY_CODE","YEAR"],left_on=["iso3c","year"])
del newdata['YEAR']
del newdata['COUNTRY_CODE']
print(newdata.shape)
newdata = newdata[~(newdata.bmkp_o.isnull())]
print(newdata.shape)
newdata = newdata[newdata.nrp_fg_o.notnull()]
print(newdata.shape)
newdata.head()

(810, 75)
(770, 75)
(696, 75)


,country,year,trade,foodsec,bmkp_o,bmkp_a,forex_o,forex_a,bpc_ac_o,bpc_ac_a,bpc_trsp_o,bpc_trsp_a,bpc_mrg_o,bpc_mrg_a,bpc_prs_o,bpc_prs_a,bpc_hnd_o,bpc_hnd_a,bpc_taxs_o,bpc_taxs_a,bpc_other_o,bpc_other_a,pcp,pcfg_ac_o,pcfg_ac_a,pcfg_trsp_o,pcfg_trsp_a,pcfg_mrg_o,pcfg_mrg_a,pcfg_prs_o,pcfg_prs_a,pcfg_hnd_o,pcfg_hnd_a,pcfg_taxs_o,pcfg_taxs_a,pcfg_other_o,pcfg_other_a,fgp,bot,bpc_qt,bpc_ql,pcfg_qt,pcfg_ql,pcrt_qt,pcrt_ql,bmkp_lcu_o,bmkp_lcu_a,r_pcp_o,r_pcp_a,r_fgp_o,r_fgp_a,pg_pc_o,pg_pc_a,pg_fg_o,pg_fg_a,pg_rt_o,pg_rt_a,nrp_pc_o,nrp_pc_a,nrp_fg_o,nrp_fg_a,nra_o,nra_a,img,forexg,acg_pc,acg_fg,mdg,mdg_percent_o,mdg_percent_a,iso3c,item_en,item_fr,item_code,EXCH
0,Armenia,2005,m,NaN,310.242976,NaN,457.686941,NaN,20.318467,NaN,8.127387,NaN,NaN,NaN,NaN,NaN,12.191080,NaN,NaN,NaN,NaN,NaN,NaN,35.218676,NaN,8.127387,NaN,27.091289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,301.516141,NaN,1.0,1.0,1,1,NaN,NaN,NaN,NaN,330.561443,NaN,295.342767,NaN,NaN,NaN,6.173374,NaN,NaN,NaN,NaN,NaN,2.090240,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARM,Apples,NaN,515,457.686941
1,Armenia,2006,m,NaN,337.039688,NaN,416.040370,NaN,23.000610,NaN,9.200244,NaN,NaN,NaN,NaN,NaN,13.800366,NaN,NaN,NaN,NaN,NaN,NaN,39.867725,NaN,9.200244,NaN,30.667481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,447.072000,NaN,1.0,1.0,1,1,NaN,NaN,NaN,NaN,360.040299,NaN,320.172574,NaN,NaN,NaN,126.899425,NaN,NaN,NaN,NaN,NaN,39.634696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARM,Apples,NaN,515,416.040370
2,Armenia,2007,m,NaN,431.542167,NaN,342.079116,NaN,29.204433,NaN,11.681773,NaN,NaN,NaN,NaN,NaN,17.522660,NaN,NaN,NaN,NaN,NaN,NaN,50.621016,NaN,11.681773,NaN,38.939243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,543.733865,0.0,1.0,1.0,1,1,NaN,NaN,NaN,NaN,460.746599,NaN,410.125583,NaN,NaN,NaN,133.608282,NaN,NaN,NaN,NaN,NaN,32.577407,NaN,32.577407,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARM,Apples,NaN,515,342.079116
3,Armenia,2008,m,NaN,520.628967,NaN,305.969400,NaN,35.589660,NaN,14.235864,NaN,NaN,NaN,NaN,NaN,21.353796,NaN,NaN,NaN,NaN,NaN,NaN,61.688744,NaN,14.235864,NaN,47.452880,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,516.391508,0.0,1.0,1.0,1,1,NaN,NaN,NaN,NaN,556.218627,NaN,494.529883,NaN,NaN,NaN,21.861625,NaN,NaN,NaN,NaN,NaN,4.420688,NaN,4.420688,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARM,Apples,NaN,515,305.969400
4,Armenia,2009,m,NaN,387.394323,NaN,363.283286,NaN,30.993952,NaN,12.397581,NaN,NaN,NaN,NaN,NaN,18.596371,NaN,NaN,NaN,NaN,NaN,NaN,53.722850,NaN,12.397581,NaN,41.325269,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,388.126857,0.0,1.0,1.0,1,1,NaN,NaN,NaN,NaN,418.388274,NaN,364.665425,NaN,NaN,NaN,23.461433,NaN,NaN,NaN,NaN,NaN,6.433687,NaN,6.433687,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARM,Apples,NaN,515,363.283286


In [7]:
col= ['bmkp_o','bmkp_a','bpc_ac_o','bpc_ac_a','bpc_trsp_o','bpc_trsp_a','bpc_mrg_o','bpc_mrg_a',
     'bpc_prs_o','bpc_prs_a','bpc_hnd_o','bpc_hnd_a','bpc_taxs_o','bpc_taxs_a','bpc_other_o','bpc_other_a','pcp','pcfg_ac_o',
     'pcfg_ac_a', 'pcfg_trsp_o', 'pcfg_trsp_a','pcfg_mrg_o','pcfg_mrg_a','pcfg_prs_o','pcfg_prs_a','pcfg_hnd_o','pcfg_hnd_a',
     'pcfg_taxs_o','pcfg_taxs_a','pcfg_other_o','pcfg_other_a','fgp']

In [8]:
for c in col:
    newdata[c] = newdata[c]*newdata['EXCH']

In [9]:
# Malawi-tobacco data is in USD
mwi_tob = data[(data['country']=='Malawi') & (data['item_en']=='Tobacco')]

for c in col:
    mwi_tob[c] = mwi_tob[c]*mwi_tob['EXCH']

mwi_tob_e = data[~((data['country']=='Malawi') & (data['item_en']=='Tobacco'))]
data = mwi_tob_e.append(mwi_tob)    

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1159059731.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mwi_tob[c] = mwi_tob[c]*mwi_tob['EXCH']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1159059731.py:8: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  data = mwi_tob_e.append(mwi_tob)


In [10]:
moz_chick = data[(data['country']=='Mozambique') & (data['item_en']=='Chicken')]

for c in col:
    moz_chick[c] = moz_chick[c]*1000

moz_chick_e = data[~((data['country']=='Mozambique') & (data['item_en']=='Chicken'))]
data = moz_chick_e.append(moz_chick)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\2698694362.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  moz_chick[c] = moz_chick[c]*1000
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\2698694362.py:7: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  data = moz_chick_e.append(moz_chick)


In [11]:
moz_chicken = data[(data['country']=='Mozambique') & (data['item_en']=='Chicken') & (data['year']==2010)]
moz_chicken['pcfg_qt'] = .72

moz_chicken_e = data[~((data['country']=='Mozambique') & (data['item_en']=='Chicken') & (data['year']==2010))]

data = moz_chicken_e.append(moz_chicken)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1082616174.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  moz_chicken['pcfg_qt'] = .72
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1082616174.py:6: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  data = moz_chicken_e.append(moz_chicken)


In [12]:
data_c = data.append(newdata)
print(data.shape)
print(newdata.shape)
data_c.shape

(1723, 75)
(696, 75)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1125282909.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  data_c = data.append(newdata)


(2419, 75)

In [13]:
data_c = data_c.fillna(0)
data_c['NOTE_1'] = 1
data_c['NOTE_10'] = 10

data_c['bmk_c'] = np.where(data_c['forex_o']==1,'LCU','USD')

# Zimbabwe data are all in USD and we will not use exchange rate from WDI for Zimbabwe only
data_c['bmk_c'] = np.where(data_c.country=='Zimbabwe', 'LCU', data_c['bmk_c'])

# # Malawi-tobacco data is in USD
# data_c['bmk_c'] = np.where( ((data_c.country=='Malawi') & (data_c.item_en=='Tobacco')), 'USD', data_c['bmk_c'])

data_c.shape

(2419, 78)

In [14]:
# Observed Benchmark Price in LCU computed
data_new = data_c[data_c.iso3c.isin(['BLR','ARM','GEO','AZE','MDA','KGZ'])]
data_e = data_c[~data_c.iso3c.isin(['BLR','ARM','GEO','AZE','MDA','KGZ'])]

data_e['bmkp_lcu_mt'] = np.where(data_e['bmk_c']=="USD",data_e['bmkp_o']*data_e['EXCH'],data_e['bmkp_o']*data_e['forex_o'])

data_new['bmkp_lcu_mt'] = data_new['bmkp_o']

data_c = data_e.append(data_new)

data_c.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\2242315674.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_e['bmkp_lcu_mt'] = np.where(data_e['bmk_c']=="USD",data_e['bmkp_o']*data_e['EXCH'],data_e['bmkp_o']*data_e['forex_o'])
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\2242315674.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_new['bmkp_lcu_mt'] = data_new['bmkp_o']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\2242315674.py:9: FutureWarning: The frame.append method is depre

(2419, 79)

In [15]:
# Processing Costs from Border to PoC computed
data_c['bpc_ac_mt'] = data_c['bpc_mrg_o'] + data_c['bpc_prs_o']

# Transport Costs from Border to Point of Competition Computed
data_c['bpc_trsp_mt'] = data_c['bpc_trsp_o'] + data_c['bpc_hnd_o']

# Other Costs from Border to PoC Computed
data_c['bpc_other_mt'] = data_c['bpc_taxs_o'] + data_c['bpc_other_o']


# Excess Costs to reconcile from Border to POC PRICE
data_c['bpc_exc'] = data_c['bpc_ac_o']-(data_c['bpc_trsp_o']+data_c['bpc_mrg_o']+data_c['bpc_prs_o']+data_c['bpc_hnd_o']+data_c['bpc_taxs_o']+data_c['bpc_other_o'])

# Observed Access Costs from Border to PoC Computed
data_c['bpc_aco_mt'] = data_c['bpc_ac_mt'] + data_c['bpc_trsp_mt'] + data_c['bpc_other_mt'] + data_c['bpc_exc'] # add excess costs to reconcile

# Observed Reference Price at the point of competition computed
adj_rp = data_c['bmkp_lcu_mt']*data_c['bpc_qt']*data_c['bpc_ql']
data_c['r_pcp_mt'] = np.where(data_c.trade=="x",adj_rp-data_c["bpc_aco_mt"],adj_rp+data_c["bpc_aco_mt"])

# Processing Costs from Point of Competition to Farm Gate Computed
data_c['pcfg_prs_mt'] = data_c['pcfg_mrg_o'] + data_c['pcfg_prs_o']

# Transport Costs from Point of Competition to Farm Gate Computed
data_c['pcfg_trsp_mt'] = data_c['pcfg_trsp_o'] + data_c['pcfg_hnd_o']

# Other Costs from Point of Competition to Farm Gate Computed
data_c['pcfg_other_mt'] = data_c['pcfg_taxs_o'] + data_c['pcfg_other_o']

# Excess Costs to reconcile from Border to POC PRICE
data_c['pcfg_exc'] = data_c['pcfg_ac_o']-(data_c['pcfg_trsp_o']+data_c['pcfg_mrg_o']+data_c['pcfg_prs_o']+data_c['pcfg_hnd_o']+data_c['pcfg_taxs_o']+data_c['pcfg_other_o'])

# Observed Access Costs from Point of Competition to Farm Gate Computed
data_c['pcfg_ac_mt'] = data_c['pcfg_prs_mt']+data_c['pcfg_trsp_mt']+data_c['pcfg_other_mt']+data_c['pcfg_exc'] # excess cost to reconcile

# Observed Reference Price at the farm gate computed
data_c['r_fgp_c'] = data_c['r_pcp_mt'] * data_c['pcfg_qt']*data_c['pcfg_ql']-data_c['pcfg_ac_mt']
# data_c.to_csv('data_check4.csv')

In [16]:
data_c['item_code'] = data_c['item_code'].replace(9999,108) #Convert teff to

data_c['nrp_fg_o'] = 1+data_c['nrp_fg_o']/100

data_c = data_c.rename(columns=mapping)

trade_map = {'x':'eXports','m':'iMports'}
data_c.TRADE_STATUS = data_c.TRADE_STATUS.map(trade_map)

data_c['RP_POC_PM'] = data_c['bpc_ac_o']+ data_c['bpc_mrg_o']+data_c['bpc_prs_o']+data_c['bpc_hnd_o']+data_c['bpc_taxs_o']+data_c['RP_POC_OC']
data_c['POC_FARM_PM'] = data_c['pcfg_ac_o']+data_c['pcfg_mrg_o']+data_c['pcfg_hnd_o']+data_c['pcfg_taxs_o']+data_c['POC_FARM_OC']
# data.to_csv('data_check5.csv')

In [17]:
# Replacing commodity code 218 with 217. The commodity name is cashew nuts, with shell 
data_c['COMMODITY_CODE'].replace(218, 217,inplace=True)
# Replacing commodity code 497 with 571. The commodity name is Mango
data_c['COMMODITY_CODE'].replace(497, 571,inplace=True)

# Replacing commodity code 241 with 242. The commodity name is Groundnuts, with shell
data_c['COMMODITY_CODE'].replace(241, 242,inplace=True)

# Replacing commodity code 329 with 328. The commodity name is seed cotton
data_c['COMMODITY_CODE'].replace(329, 328,inplace=True)

senbgd = data_c[(data_c['COUNTRY_LABEL'].isin(["Senegal", "Bangladesh"])) & (data_c['COMMODITY_CODE']==402)]
senbgd['COMMODITY_CODE'].replace(402, 403,inplace=True)
senbgd_e = data_c[~((data_c['COUNTRY_LABEL'].isin(["Senegal", "Bangladesh"])) & (data_c['COMMODITY_CODE']==402))]
data_c = senbgd_e.append(senbgd)
print(data_c.shape)

(2419, 93)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1743326549.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  senbgd['COMMODITY_CODE'].replace(402, 403,inplace=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1743326549.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  data_c = senbgd_e.append(senbgd)


In [18]:
mlw = data_c[(data_c.COUNTRY_LABEL=='Malawi') & (data_c.COMMODITY_LABEL=='Groundnuts, with shell')]
# mlw.to_csv('malawi_groundnuts.csv')

In [20]:
### PRODUCTION DATA
production = pd.read_csv(os.path.join(base_dir,"Production_Crops_Livestock_E_All_Data_(Normalized).csv"),encoding="latin-1")
production = production[production.Element=='Production']
production = production.replace("United Republic of Tanzania","Tanzania")
production = production.replace("Seed cotton, unginned","Seed cotton")
production.head()

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1205772819.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  production = pd.read_csv(os.path.join(base_dir,"Production_Crops_Livestock_E_All_Data_(Normalized).csv"),encoding="latin-1")


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
111,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,1961,1961,t,0.0,A,NaN
112,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,1962,1962,t,0.0,A,NaN
113,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,1963,1963,t,0.0,A,NaN
114,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,1964,1964,t,0.0,A,NaN
115,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,1965,1965,t,0.0,A,NaN


In [21]:
cattle = production[production.Item=="Meat of cattle with the bone, fresh or chilled"]
cattle['Item Code'] = 866
cattle.head()

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1631733538.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cattle['Item Code'] = 866


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
4187,2,'004,Afghanistan,866,'21111.01,"Meat of cattle with the bone, fresh or chilled",5510,Production,1961,1961,t,43000.0,E,NaN
4188,2,'004,Afghanistan,866,'21111.01,"Meat of cattle with the bone, fresh or chilled",5510,Production,1962,1962,t,45800.0,E,NaN
4189,2,'004,Afghanistan,866,'21111.01,"Meat of cattle with the bone, fresh or chilled",5510,Production,1963,1963,t,47250.0,E,NaN
4190,2,'004,Afghanistan,866,'21111.01,"Meat of cattle with the bone, fresh or chilled",5510,Production,1964,1964,t,48000.0,E,NaN
4191,2,'004,Afghanistan,866,'21111.01,"Meat of cattle with the bone, fresh or chilled",5510,Production,1965,1965,t,48700.0,E,NaN


In [22]:
import re

print(production['Item Code'][production.Item=='Cashew nuts, in shell'].unique())
# print(production['Item Code'][production.Item=='Cashew nuts, without shell'].unique())

print(production['Item Code'][production.Item=='Beans, dry'].unique())
print(production['Item Code'][production.Item=='Cereals n.e.c.'].unique())
print(production['Item Code'][production.Item=='Onions and shallots, dry (excluding dehydrated)'].unique())
print(production['Item Code'][production.Item=='Onions and shallots, green'].unique())
# production.Item.unique()

[217]
[176]
[108]
[403]
[402]


In [23]:
cashew = production[production["Item Code"]==217]
cashew["Item Code"] = 2171
cashew["Value"]= cashew.Value*.2
cashew["Item"] = "Cashew nuts, without shell"
cashew = cashew[cashew.Unit=="t"]
cashew[(cashew.Item=='Cashew nuts, without shell') & (cashew.Area=='Mozambique') & (cashew.Year>=2005)].head()

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1223360849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cashew["Item Code"] = 2171
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1223360849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cashew["Value"]= cashew.Value*.2
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\1223360849.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the

,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
1929715,144,'508,Mozambique,2171,'01372,"Cashew nuts, without shell",5510,Production,2005,2005,t,20867.4,A,NaN
1929716,144,'508,Mozambique,2171,'01372,"Cashew nuts, without shell",5510,Production,2006,2006,t,12564.2,A,NaN
1929717,144,'508,Mozambique,2171,'01372,"Cashew nuts, without shell",5510,Production,2007,2007,t,14879.0,A,NaN
1929718,144,'508,Mozambique,2171,'01372,"Cashew nuts, without shell",5510,Production,2008,2008,t,17000.0,A,NaN
1929719,144,'508,Mozambique,2171,'01372,"Cashew nuts, without shell",5510,Production,2009,2009,t,12800.0,A,NaN


In [24]:
#print(cashew)
production = production[production.Unit=="t"]
production = pd.concat([production,cattle])
production = pd.concat([production,cashew])

production['PRODQ'] = production.Value
production['PRODQ_PHY_UNIT'] = production.Unit
production['PRODQ_PHY_UNIT'] = production.PRODQ_PHY_UNIT.replace("t","MT")

# Replacing commodity 'Onions, shallots, green' with 'Onions'. Both commodities have same commodity code, 402 
production['Item'].replace('Onions and shallots, green', 'Onions',inplace=True)
production['Item'].replace('Groundnuts, excluding shelled', 'Groundnuts, with shell',inplace=True)
production['Item'].replace('Raw milk of cattle', 'Milk, cow',inplace=True)
production['Item'].replace('Meat of chickens, fresh or chilled', 'Chicken',inplace=True)
print(production['Item Code'][production.Item=='Groundnuts, with shell'].unique())


[242]


In [25]:
# missing_prodq = pd.read_excel(os.path.join(base_dir,'MAFAP_PI_consortium_dataset_Missing_Data_2021 obs.xlsx'))                   

In [26]:
m = production.merge(data_c,how="right",left_on = ["Area","Year","Item Code"], right_on=["COUNTRY_LABEL","YEAR","COMMODITY_CODE"])
m["COMMODITY_CODE"] = m["COMMODITY_CODE"].replace(108,2005) # teff
m["COMMODITY_CODE"] = m["COMMODITY_CODE"].replace(2171,230) # cashews, without shells
m.shape

(2419, 109)

In [27]:
# plugging missing production quantity data
# m = m.merge(missing_prodq, how='left')
# m.PRODQ = np.where(((m.PRODQ.isnull()) & (m.YEAR==2021)), m.pq, m.PRODQ)
m.PRODQ_PHY_UNIT = np.where(m.PRODQ_PHY_UNIT.isnull(),'MT',m.PRODQ_PHY_UNIT)
m.shape

(2419, 109)

In [28]:
m["MPS"] = (m.PROP-m.REFP)*m.PRODQ

m.rename(columns={'EXCH':'ER_OFFICIAL'}, inplace=True)
m["COMMODITY_CODE"] = m["COMMODITY_CODE"].map(int)

print(m.shape)

(2419, 110)


In [29]:
# Drop cashew nuts without shell
m = m[m.COMMODITY_CODE!=230]
print(m.shape)
m = m[m.PROP!=0]
print(m.shape)
m = m[~(m.REFP<0)]
m.shape

(2404, 110)
(2404, 110)


(2404, 110)

In [30]:
def recompute_cattle_price(data,rows,factor):
	rows['PROP'] = rows.PROP/factor
	rows['REFP'] = rows.REFP/factor
	data.loc[rows.index] = rows
	return data

eth = m['COUNTRY_CODE']=="ETH"
mli = m['COUNTRY_CODE']=="MLI"
uga = m['COUNTRY_CODE']=="UGA"
cow = m['COMMODITY_LABEL'] == "Cattle"

eth_cow = m[eth & cow]
uga_cow = m[uga & cow]
mli_cow = m[mli & cow]

m = recompute_cattle_price(m,eth_cow,.11)
m = recompute_cattle_price(m,uga_cow,.11)
m = recompute_cattle_price(m,mli_cow,.13)
m.shape


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\3960745042.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rows['PROP'] = rows.PROP/factor
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_34092\3960745042.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rows['REFP'] = rows.REFP/factor


(2404, 110)

In [31]:
local_currency = {
    'BGD':'BDT',
    'BDI':'BIF',
    'BEN':'XOF',
    'BFA':'XOF',
    'ETH':"ETB",
    'GHA':'GHS',
    'KEN':'KES',
    'MLI':'XOF',
    'MOZ':'MZN',
    'MWI':'MWK',
    'NGA':'NGN',
    'RWA':'RWF',
    'SEN':'XOF',
    'TZA':'TZS',
    'UGA':'UGX',
    'ARM':'AMD', 
    'AZE':'AZN',
    'BLR':'BYN',
    'GEO':'GEL',
    'KGZ':'KGS',
    'MDA':'MDL',
    'ZMB':'ZMW',
    'ZWE':'USD'
}

In [33]:
m['SOURCE'] = "FAO"
date = datetime.strftime(datetime.today(),"%m/%d/%Y")
m['TIMESTAMP'] = date
m['EROFFICIAL_SOURCE'] = 'WDI'
m['EROFFICIAL_UNIT'] = m.COUNTRY_CODE.map(local_currency)
m['SOURCEFILE_DATE'] = '12/06/2024'
m['SOURCE_FILE'] = 'PI_Consortium_dataset_2024-11-27.xlsx'
m['EX_SOURCEFILE']= 'exchange_rate_2024.csv'

m['PROP_MON_UNIT'] = m.EROFFICIAL_UNIT
m['REFP_MON_UNIT'] = m.EROFFICIAL_UNIT
m['PROP_PHY_UNIT'] = 'MT'
m['REFP_PHY_UNIT'] = 'MT'

# Here we are using exchange rate directly from FAO dataset
m['ER_OFFICIAL'] = np.where(m['COUNTRY_CODE']=='ZWE', 1, m['ER_OFFICIAL'])
m['EROFFICIAL_SOURCE'] = np.where(m['COUNTRY_CODE']=='ZWE', 'FAO', m['EROFFICIAL_SOURCE'])
# m = m[m.ER_OFFICIAL!=0]

m = m[["COUNTRY_LABEL","COUNTRY_CODE","COMMODITY_LABEL","COMMODITY_CODE","YEAR","PRODQ","BP","PROP","REFP","MPS","PSCT","NPC_SOURCE","RP_POC_TM","RP_POC_OC",
	"POC_FARM_OC","POC_FARM_TM","POCPRICE","RP_POC_PM","POC_FARM_PM","RP_POC_QADJ","RP_POC_QLTADJ","POC_FARM_QADJ","POC_FARM_QLTADJ",
	"ER_USED","ER_OFFICIAL","EROFFICIAL_UNIT","EROFFICIAL_SOURCE","PRODQ_PHY_UNIT","PROP_MON_UNIT","PROP_PHY_UNIT","REFP_MON_UNIT","REFP_PHY_UNIT",
	"TRADE_STATUS","SOURCE","SOURCE_FILE","SOURCEFILE_DATE","EX_SOURCEFILE","TIMESTAMP", "NOTE_1","NOTE_10"]]

m.to_csv("MAFAP_input_file_2025.csv", index=False)

m.shape

(2404, 40)

In [ ]:
# import matplotlib.cm as cm, matplotlib.font_manager as fm
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

# df=m[['COUNTRY_LABEL','YEAR','COMMODITY_LABEL','PROP', 'REFP']]
# df['lnPROP'] = np.log(df['PROP'])
# df['lnREFP'] = np.log(df['REFP'])

# sns.color_palette("bright")
# sns.set(font_scale=1.5)
# sns.set_style("white")
# g=sns.FacetGrid(df, col="COMMODITY_LABEL",col_wrap=3,size=4,sharey=True,aspect=1.2)
# g.set(xmargin=0.05, ymargin=0.15)
# g.map(plt.plot, "YEAR", "lnPROP", 
#       color='k',
#       lw=1.25,
#       marker='.', 
#       mfc='r',
#       markevery=[-1],
#       ms=10)

# g.set_xticklabels(rotation=90,fontsize=10)
# g.set_titles("{col_name}")
# g.set_xlabels("")
# g.set_ylabels("")
# plt.subplots_adjust(top=0.9)
# g.fig.suptitle('Producer price over the years by commodities',fontsize=30)